# Metrics for text, translation & speech

_Metrics & Evaluation — measuring models, data & statistics_

**How to put a number on a machine's words — comparing what it wrote to what a human wrote.**

How do you grade a sentence a machine wrote? You compare it to a reference — a sentence a human wrote that you trust. The metric turns "how close are these two sentences?" into a number.

       There are three big families, and the whole lesson hangs on this split:

---

This notebook is a practice scaffold for the **Metrics for text, translation & speech** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — sacrebleu, rouge_score, evaluate, jiwer, torchmetrics

In [ ]:
# pip install sacrebleu rouge-score evaluate jiwer torchmetrics torch
# --- BLEU, chrF, TER (sacrebleu: the standard, reproducible implementation) ---
import sacrebleu
refs = [["the quick brown fox jumps over the lazy dog"]]   # list of reference lists
cand = ["a quick brown fox jumped over the lazy dog"]
print("BLEU :", sacrebleu.corpus_bleu(cand, refs).score)     # 0..100, higher better
print("chrF :", sacrebleu.corpus_chrf(cand, refs).score)     # character F-score
print("TER  :", sacrebleu.corpus_ter(cand, refs).score)      # edit rate, lower better

# --- ROUGE-1 / ROUGE-2 / ROUGE-L (summarization overlap/recall) ---
from rouge_score import rouge_scorer
sc = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
print(sc.score("the quick brown fox jumps over the lazy dog",
               "a quick brown fox jumped over the lazy dog"))

# --- HuggingFace evaluate: one hub for many metrics (incl. METEOR, BERTScore) ---
import evaluate
meteor = evaluate.load("meteor")
print("METEOR:", meteor.compute(predictions=cand, references=[r[0] for r in refs]))
bertscore = evaluate.load("bertscore")                       # embedding-based, semantic
print("BERTScore:", bertscore.compute(predictions=cand,
      references=[r[0] for r in refs], lang="en")["f1"])
squad = evaluate.load("squad")                               # exact match + token-F1 for QA
print(squad.compute(
    predictions=[{"id": "1", "prediction_text": "Paris"}],
    references=[{"id": "1", "answers": {"text": ["Paris"], "answer_start": [0]}}]))

# --- WER and CER for speech / OCR (jiwer) ---
import jiwer
print("WER:", jiwer.wer("turn the lights on now", "turn lights on now"))   # 0.2
print("CER:", jiwer.cer("turn the lights on now", "turn lights on now"))

# --- torchmetrics.text: Perplexity, BLEU, ROUGE, BERTScore in a PyTorch pipeline ---
import torch
from torchmetrics.text import Perplexity, BLEUScore, ROUGEScore, BERTScore
print("BLEU (tm):", BLEUScore()(cand, [[r[0] for r in refs]]))
print("ROUGE (tm):", ROUGEScore()(cand, [r[0] for r in refs]))
# Perplexity wants per-token logits (V = vocab size) and the true next-token ids:
logits = torch.randn(2, 8, 100)          # (batch, seq_len, vocab) model outputs
target = torch.randint(0, 100, (2, 8))   # (batch, seq_len) gold tokens
print("Perplexity:", Perplexity()(logits, target))   # exp of mean cross-entropy
print("BERTScore (tm):", BERTScore()(cand, [r[0] for r in refs])["f1"])

## Visualize the data & results

_Given one human reference and three machine candidates, do the metrics agree on which candidate is best — high BLEU and ROUGE, low WER for the good one?_

In [ ]:
import numpy as np
from collections import Counter

reference = "the quick brown fox jumps over the lazy dog"
candidates = {
    "A": "the quick brown fox jumps over the lazy dog",   # exact copy
    "B": "a quick brown fox jumped over the lazy dog",     # two word changes
    "C": "the dog ran past a brown fox",                   # mostly different
}

def tokenize(s):
    return s.split()

def wer(ref, hyp):                       # Word Error Rate via Levenshtein edit distance
    r, h = tokenize(ref), tokenize(hyp)
    n, m = len(r), len(h)
    D = np.zeros((n + 1, m + 1), dtype=int)
    D[:, 0] = np.arange(n + 1)           # deleting all reference words
    D[0, :] = np.arange(m + 1)           # inserting all hypothesis words
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if r[i - 1] == h[j - 1] else 1
            D[i, j] = min(D[i - 1, j] + 1,        # deletion
                          D[i, j - 1] + 1,        # insertion
                          D[i - 1, j - 1] + cost) # match or substitution
    return D[n, m] / max(n, 1)

def clipped_overlap(ref, hyp):           # shared unigrams, each clipped to ref count
    rc, hc = Counter(tokenize(ref)), Counter(tokenize(hyp))
    return sum(min(hc[w], rc[w]) for w in hc)

def bleu1(ref, hyp):                     # BLEU = clipped unigram precision here
    overlap = clipped_overlap(ref, hyp)
    return overlap / max(len(tokenize(hyp)), 1)

def rouge1_f1(ref, hyp):                 # ROUGE-1 = F1 of unigram precision & recall
    overlap = clipped_overlap(ref, hyp)
    rec = overlap / max(len(tokenize(ref)), 1)
    prec = overlap / max(len(tokenize(hyp)), 1)
    return 0.0 if rec + prec == 0 else 2 * rec * prec / (rec + prec)

for name, cand in candidates.items():
    print(name,
          "BLEU-1=%.3f" % bleu1(reference, cand),
          "ROUGE-1=%.3f" % rouge1_f1(reference, cand),
          "WER=%.3f" % wer(reference, cand))
# A BLEU-1=1.000 ROUGE-1=1.000 WER=0.000
# B BLEU-1=0.778 ROUGE-1=0.778 WER=0.222
# C BLEU-1=0.571 ROUGE-1=0.500 WER=0.889

import matplotlib.pyplot as plt
names = list(candidates)
x = np.arange(len(names))
plt.bar(x - 0.25, [bleu1(reference, candidates[n]) for n in names], 0.25, label="BLEU-1", color="#4ea1ff")
plt.bar(x + 0.00, [rouge1_f1(reference, candidates[n]) for n in names], 0.25, label="ROUGE-1", color="#7ee787")
plt.bar(x + 0.25, [wer(reference, candidates[n]) for n in names], 0.25, label="WER", color="#ffb454")
plt.xticks(x, names); plt.ylabel("score (0..1)"); plt.legend()
plt.title("BLEU-1 & ROUGE-1 up, WER down for the better candidate")
plt.show()

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Reference: cats sit on mats (4 words). Candidate: cats sit on the mat (5 words). Compute the unigram (1-gram) precision $p_1$ and ROUGE-1 recall.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- List candidate words: cats, sit, on, the, mat. Which appear in the reference {cats, sit, on, mats}? cats, sit, on do; "the" and "mat" do not. — _Precision counts candidate words found in the reference._
- $p_1 = 3/5 = 0.6$. — _3 of the 5 candidate words matched._
- Recall counts reference words produced: of {cats, sit, on, mats}, the candidate has cats, sit, on (not "mats", only "mat"). So recall $= 3/4 = 0.75$. — _"mat" $\ne$ "mats" — exact word match fails, which is the paraphrase blind spot._

**Answer:** Unigram precision $p_1 = 3/5 = 0.6$; ROUGE-1 recall $= 3/4 = 0.75$. Note "mat" vs "mats" costs a match even though the meaning is the same — the classic overlap-metric weakness.

</details>

**Problem 2.** A speech model transcribes the 5-word reference turn the lights on now as turn lights on now. What is the WER (Word Error Rate)?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Align the two: the only difference is the missing word "the". — _Find the cheapest set of single-word edits._
- That is one deletion: $S=0, D=1, I=0$. — _Deleting "the" from the reference gives the candidate._
- $\text{WER} = (S+D+I)/(\text{reference words}) = 1/5 = 0.2$. — _Divide edits by the 5 reference words to get a rate._

**Answer:** WER $= 1/5 = 0.2$ (one deleted word out of five). Lower is better, so this is a fairly good transcript.

</details>

**Problem 3.** Your translation model scores high BLEU but a reviewer says the outputs read like copies of the source. Which metric should you add, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- BLEU and ROUGE reward shared n-grams, so copying scores well — they cannot see "is this novel or just pasted?". — _Overlap metrics are blind to copying and to paraphrase quality._
- Add a meaning-aware metric (BERTScore or COMET) to check semantic adequacy, and a diversity/novelty check (distinct-n or self-BLEU) to catch copying. — _Embedding metrics judge meaning; diversity metrics judge repetition._

**Answer:** Pair BLEU with an embedding-based metric (BERTScore / COMET) for meaning, and a diversity metric (distinct-n or self-BLEU) to detect copying. No single overlap number should be the only judge.

</details>